# Task 4: final answers (train-only design)
The submitted `task4` column = **mean of two DINOv3 ViT-7B/16 + LoRA runs** (training seeds 7 and 8, same seed-42 folds), trained in `task4_dino7b_lora_seed7.ipynb` / `task4_dino7b_lora_seed8.ipynb` on a B200 GPU (15 epochs, 192×384, final EMA weights, test file = ½ mean of the 5 fold models + ½ the full-data model).

**No step of this pipeline looks at the test images** (competition rule: priors, thresholds and other parameters must be computable without seeing the test set). Test images are only predicted:
* augmentation: damage types that train itself contains, with strengths checked on train images only (Section 1);
* validation: the plain stratified 5 folds; no epoch is picked on validation (final EMA weights);
* every decision below uses the **clean** out-of-fold (OOF) predictions of the training images.

In [1]:
import json, math, os, numpy as np, pandas as pd, torch, torch.nn.functional as F
from PIL import Image
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
NAMES = ['7-11 Knights', 'FiveN', 'Free Fried', 'PUBE', 'Roblock']
R, RUNS = 'results', ['_seed7', '_seed8']
train, test = pd.read_csv('train.csv'), pd.read_csv('test.csv')
y = train['label'].values; PI = np.bincount(y, minlength=5) / len(y)
folds = [b for _, b in StratifiedKFold(5, shuffle=True, random_state=42).split(y, y)]   # same folds as the training notebooks
f1 = lambda p, m=slice(None), a=0.: f1_score(y[m], (p[m] * PI ** -a).argmax(1), average='macro')
OOF = {s: np.load(f'{R}/task4_oof_dino7b_clean{s}.npy') for s in RUNS}

## 1. Train-only check of the augmentation
The training notebooks augment with the damage seen in the data (rotation/shift with black corners, hue shift, inversion, grey, saturation, brightness/gamma, blur, noise, occlusion). To check how close the augmented images stay to the damage train already has, 3,000 train images are measured before and after one random augmentation (the training notebook's own `corrupt` function, run on CPU). Test images are not opened.

In [2]:
src = next(''.join(c['source']) for c in json.load(open('task4_dino7b_lora_seed7.ipynb', encoding='utf-8'))['cells'] if 'def corrupt' in ''.join(c['source']))
ns = {'torch': torch, 'F': F, 'math': math}
exec(src[src.index('YIQ = torch.tensor'):src.index('class Imgs')] + src[src.index('def corrupt'):src.index('def prep(')], ns)   # the exact training code
YIQ = ns['YIQ'].numpy()

def damage(x):   # x: (n, 3, H, W) in [0, 1] -> per-image damage statistics
    out = []
    for a in x.permute(0, 2, 3, 1).numpy():
        yiq = a @ YIQ.T; Y = yiq[..., 0]; k = 6
        corners = np.concatenate([a[:k, :k], a[:k, -k:], a[-k:, :k], a[-k:, -k:]]).reshape(-1, 3)
        out.append({'black corners': (corners.max(1) < .06).mean(), 'brightness': Y.mean(), 'saturation': np.abs(a - a.mean(2, keepdims=True)).mean(),
                    'colour cast': np.hypot(yiq[..., 1].mean(), yiq[..., 2].mean()),
                    'noise': np.abs(4 * Y[1:-1, 1:-1] - Y[:-2, 1:-1] - Y[2:, 1:-1] - Y[1:-1, :-2] - Y[1:-1, 2:]).mean()})
    return pd.DataFrame(out)

idx = np.random.default_rng(0).choice(len(train), 3000, replace=False)
x = torch.stack([torch.from_numpy(np.array(Image.open(f'train/{train.file_name[i]}').convert('RGB'))).permute(2, 0, 1) for i in idx]).float() / 255
g = torch.Generator(); g.manual_seed(0)
raw, aug = damage(x), damage(torch.cat([ns['corrupt'](b.clone(), g, hud=.5) for b in x.split(500)]))
q = lambda d: d.quantile([.01, .5, .99]).T.round(3).astype(str).agg(' / '.join, axis=1)
lo, hi = raw.quantile(.01), raw.quantile(.99)
print(pd.DataFrame({'train 1% / 50% / 99%': q(raw), 'augmented 1% / 50% / 99%': q(aug),
                    'augmented inside train 1-99%': ((aug >= lo) & (aug <= hi)).mean().round(3)}).to_string())

                train 1% / 50% / 99% augmented 1% / 50% / 99%  augmented inside train 1-99%
black corners    0.0 / 0.264 / 0.681        0.0 / 0.271 / 1.0                         0.787
brightness     0.063 / 0.443 / 0.865      0.036 / 0.33 / 0.87                         0.955
saturation     0.008 / 0.068 / 0.286       0.0 / 0.06 / 0.295                         0.923
colour cast      0.0 / 0.049 / 0.333       0.0 / 0.03 / 0.335                         0.889
noise          0.056 / 0.253 / 0.451    0.011 / 0.111 / 0.291                         0.770


**Reading.** Brightness, saturation and colour cast of the augmented images stay inside train's own 1–99% range for 89–96% of the images, so these strengths copy damage that train already has. Two differences: the augmented images are **smoother** than train (the noise statistic drops: resampling for the rotation averages neighbouring pixels, and the added noise is milder than train's own), and they get **black corners more often** (rotation plus darkening). The augmentation is kept as it is.

## 2. The two runs and their mean (clean OOF)
Both seeds were trained to be averaged (a seed ensemble); the table only reports how much the average helps, per fold.

In [3]:
mean2 = np.mean(list(OOF.values()), 0)
rows = {**{f'seed {s[-1]}': OOF[s] for s in RUNS}, 'mean of the two': mean2}
print(pd.DataFrame({k: {'clean OOF macro-F1': round(f1(p), 5), 'wrong images': int((p.argmax(1) != y).sum()),
                        'wrong per fold': [int((p[b].argmax(1) != y[b]).sum()) for b in folds]} for k, p in rows.items()}).T.to_string())

                clean OOF macro-F1 wrong images        wrong per fold
seed 7                     0.99711           81  [17, 11, 18, 18, 17]
seed 8                      0.9972           75  [16, 16, 16, 15, 12]
mean of the two            0.99748           69  [13, 12, 15, 16, 13]


## 3. Decision rule: `argmax p · π^−α`
π = training class shares; α > 0 divides out part of the training prior, which can help the small classes under macro-F1. α is chosen on the clean OOF only: an α > 0 is used only if it beats plain argmax on at least 4 of the 5 folds, and then the one with the best OOF macro-F1.

In [4]:
ALPHAS = [0, .25, .5, .75, 1.]
tab = pd.DataFrame({'alpha': ALPHAS, 'clean OOF macro-F1': [f1(mean2, a=a) for a in ALPHAS],
                    'folds better than argmax': [sum(f1(mean2, b, a) > f1(mean2, b, 0) for b in folds) for a in ALPHAS]})
print(tab.round(5).to_string(index=False))
ok = tab[(tab.alpha > 0) & (tab['folds better than argmax'] >= 4)]
ALPHA = float(ok.loc[ok['clean OOF macro-F1'].idxmax(), 'alpha']) if len(ok) else 0.
print('alpha used:', ALPHA)
print('per-class F1 (clean OOF):', dict(zip(NAMES, f1_score(y, (mean2 * PI ** -ALPHA).argmax(1), average=None).round(4))))

 alpha  clean OOF macro-F1  folds better than argmax
  0.00             0.99748                         0
  0.25             0.99746                         3
  0.50             0.99736                         3
  0.75             0.99723                         1
  1.00             0.99697                         0
alpha used: 0.0
per-class F1 (clean OOF): {'7-11 Knights': np.float64(0.9985), 'FiveN': np.float64(0.9969), 'Free Fried': np.float64(0.9979), 'PUBE': np.float64(0.9977), 'Roblock': np.float64(0.9964)}


## 4. Test predictions → `submission_task4.csv` and the `task4` column of `../submission.csv`

In [5]:
pt = np.mean([np.load(f'{R}/task4_test_probs_dino7b{s}.npy') for s in RUNS], 0)
labels = (pt * PI ** -ALPHA).argmax(1)
pd.DataFrame({'id': test['id'], 'task4': labels}).to_csv('submission_task4.csv', index=False)
if os.path.exists('../submission.csv'):   # put task4 into the full file, matched by ANS id
    sub = pd.read_csv('../submission.csv'); new = sub['id'].map(dict(zip(test['id'], labels)))
    assert new.notna().all(); print('rows different from ../submission.csv task4:', (new.astype(int) != sub['task4']).sum())
    sub['task4'] = new.astype(int); sub.to_csv('../submission.csv', index=False)

rows different from ../submission.csv task4: 0
